
# Exp41 — Condition-Specific Validation-Calibrated PRR Trust

This is a **cache-only** follow-up to Exp40.

For each of the 120 forecasting conditions

\[
c=(\text{dataset},\text{horizon},\text{backbone}),
\]

we select one bounded trust coefficient using **validation only**:

\[
\beta_c^\star
=
\arg\min_{\beta\in\mathcal A}
\operatorname{MSE}_{\mathrm{val},c}(\beta),
\]

with

\[
\mathcal A=\{0,0.025,0.05,0.075,0.10,0.15,0.20\}.
\]

The selected \(\beta_c^\star\) is then frozen and applied once to the corresponding test condition.

The downstream prediction is

\[
\hat{\mathbf y}^{A}
=
\hat{\mathbf y}^{D}
+
\beta_c^\star
\left(
\hat{\mathbf y}^{R}-\hat{\mathbf y}^{D}
\right).
\]

## What this experiment does **not** do

- no PRR retraining,
- no forecaster retraining,
- no forecaster inference,
- no test-set hyperparameter selection.

If Exp40 outputs exist, its validation beta curves are reused directly.


In [ ]:

# Cell 1 — setup

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))

MANIFEST_ROOT = ROOT / "results" / "exp33" / "final_eval_manifest"
PATCHTST_DIRECT_ROOT = ROOT / "results" / "exp34" / "patchtst_direct_cache"
OTHER_DIRECT_ROOT = ROOT / "results" / "exp37D" / "direct_cache"
PRR_PART_ROOT = ROOT / "results" / "exp35B" / "channel_parts"

EXP40_ROOT = ROOT / "results" / "exp40_calibrated_prr"
EXP40_VAL_CURVE = EXP40_ROOT / "validation_beta_curve_all_conditions.csv"
EXP40_TEST120 = EXP40_ROOT / "test_120_conditions.csv"

OUT = ROOT / "results" / "exp41_condition_calibrated_prr"
OUT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)
BACKBONES = ("PatchTST","iTransformer","TimeMixer","SegMoE","DLinear")

BETA_GRID = np.array([0.0, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20], dtype=np.float64)
FIXED_BETA = 0.10
TOL = 1e-10

print("ROOT:", ROOT)
print("OUT :", OUT)
print("Exp40 validation curve:", EXP40_VAL_CURVE)
print("Beta grid:", BETA_GRID.tolist())

assert MANIFEST_ROOT.exists(), MANIFEST_ROOT
assert PRR_PART_ROOT.exists(), PRR_PART_ROOT


In [ ]:

# Cell 2 — exact Exp33 manifest loader

def load_manifest(dataset, horizon, split):
    p = MANIFEST_ROOT / f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(), p

    with np.load(p, allow_pickle=False) as z:
        a = np.asarray(z["anchor"], np.int64)
        c = np.asarray(z["channel"], np.int64)

    assert a.shape == c.shape
    return a, c

rows = []
for d in DATASETS:
    for h in HORIZONS:
        for sp in ("val", "test"):
            a, c = load_manifest(d, h, sp)
            rows.append({
                "dataset": d,
                "horizon": h,
                "split": sp,
                "pairs": len(a),
                "channels": len(np.unique(c)),
            })

manifest_df = pd.DataFrame(rows)
display(manifest_df)
assert len(manifest_df) == 48
print("[PASS] Exp33 manifests = 48/48")


In [ ]:

# Cell 3 — direct cache loader

def direct_slug(backbone):
    return backbone.lower().replace("-", "").replace("_", "")

def direct_path_candidates(backbone, dataset, horizon, split):
    root = PATCHTST_DIRECT_ROOT if backbone == "PatchTST" else OTHER_DIRECT_ROOT
    slug = direct_slug(backbone)

    exact = [
        root / f"{dataset}_H{horizon}_{split}_{slug}_direct.npz",
        root / f"{dataset}_H{horizon}_{split}_{backbone}_direct.npz",
        root / f"{dataset}_H{horizon}_{split}_direct.npz",
    ]
    out = [p for p in exact if p.exists()]

    if not out and root.exists():
        for p in root.rglob("*.npz"):
            s = p.name.lower()
            if dataset.lower() in s and f"h{horizon}" in s and split in s:
                out.append(p)

    return out

def load_direct(backbone, dataset, horizon, split):
    ma, mc = load_manifest(dataset, horizon, split)

    for p in direct_path_candidates(backbone, dataset, horizon, split):
        try:
            with np.load(p, allow_pickle=False) as z:
                required = {"anchor", "channel", "direct", "truth"}
                if not required.issubset(z.files):
                    continue

                a = np.asarray(z["anchor"], np.int64)
                c = np.asarray(z["channel"], np.int64)
                d = np.asarray(z["direct"], np.float32)
                y = np.asarray(z["truth"], np.float32)

            ok = (
                np.array_equal(a, ma)
                and np.array_equal(c, mc)
                and d.shape == (len(ma), horizon)
                and y.shape == (len(ma), horizon)
                and np.isfinite(d).all()
                and np.isfinite(y).all()
            )
            if ok:
                return {
                    "anchor": a,
                    "channel": c,
                    "direct": d,
                    "truth": y,
                    "path": str(p),
                }
        except Exception:
            pass

    raise FileNotFoundError(
        f"Exact direct cache not found: {backbone} {dataset} H={horizon} {split}"
    )

audit = []
for b in BACKBONES:
    for d in DATASETS:
        for h in HORIZONS:
            ok = True
            for sp in ("val", "test"):
                try:
                    load_direct(b, d, h, sp)
                except Exception:
                    ok = False
            audit.append({"backbone": b, "dataset": d, "horizon": h, "ok": ok})

direct_audit = pd.DataFrame(audit)
display(direct_audit.groupby("backbone", as_index=False).agg(
    total=("ok", "size"), ok=("ok", "sum")
))

assert int(direct_audit["ok"].sum()) == 120
print("[PASS] exact direct val/test cache = 120/120 conditions")


In [ ]:

# Cell 4 — PRR retrieval cache assembler

def prr_channel_file(dataset, horizon, split, ch):
    return (
        PRR_PART_ROOT
        / dataset
        / f"H{horizon}"
        / split
        / f"ch{int(ch):04d}.npz"
    )

def load_prr(dataset, horizon, split):
    ma, mc = load_manifest(dataset, horizon, split)
    out = np.full((len(ma), horizon), np.nan, dtype=np.float32)

    for ch in np.unique(mc):
        rows = np.flatnonzero(mc == ch)
        expected_anchor = ma[rows]

        p = prr_channel_file(dataset, horizon, split, int(ch))
        assert p.exists(), p

        with np.load(p, allow_pickle=False) as z:
            assert "anchor" in z.files and "retrieval" in z.files
            a = np.asarray(z["anchor"], np.int64)
            r = np.asarray(z["retrieval"], np.float32)

        assert r.shape == (len(a), horizon), (p, r.shape)

        if np.array_equal(a, expected_anchor):
            order = np.arange(len(a), dtype=np.int64)
        else:
            pos = {}
            for j, v in enumerate(a):
                pos.setdefault(int(v), []).append(j)

            order = []
            for v in expected_anchor:
                bucket = pos.get(int(v), [])
                if not bucket:
                    raise RuntimeError(
                        f"PRR cache mismatch: {dataset} H={horizon} "
                        f"{split} ch={ch}, anchor={int(v)}"
                    )
                order.append(bucket.pop(0))
            order = np.asarray(order, np.int64)

        out[rows] = r[order]

    assert out.shape == (len(ma), horizon)
    assert np.isfinite(out).all()

    return {
        "anchor": ma,
        "channel": mc,
        "retrieval": out,
    }

for d in DATASETS:
    for h in HORIZONS:
        for sp in ("val", "test"):
            load_prr(d, h, sp)

print("[PASS] exact PRR val/test retrieval = 48/48")


In [ ]:

# Cell 5 — metric helpers

def mse(a, b):
    a = np.asarray(a, np.float64)
    b = np.asarray(b, np.float64)
    return float(np.mean((a - b) ** 2))

def mae(a, b):
    a = np.asarray(a, np.float64)
    b = np.asarray(b, np.float64)
    return float(np.mean(np.abs(a - b)))

def combine(direct, retrieval, beta):
    d = np.asarray(direct, np.float32)
    r = np.asarray(retrieval, np.float32)
    return d + float(beta) * (r - d)

def gain_pct(direct_mse, method_mse):
    return 100.0 * (direct_mse - method_mse) / direct_mse

def wtl(series, tol=TOL):
    x = pd.Series(series).dropna().to_numpy(float)
    return {
        "W": int((x > tol).sum()),
        "T": int((np.abs(x) <= tol).sum()),
        "L": int((x < -tol).sum()),
    }

print("[PASS] helpers ready")


In [ ]:

# Cell 6 — obtain validation beta curves
# Fast path: reuse Exp40 output. Fallback: recompute from caches.

expected_cols = {
    "backbone", "dataset", "horizon", "beta",
    "direct_val_mse", "calibrated_val_mse",
    "mse_ratio_vs_direct", "gain_vs_direct_pct",
}

if EXP40_VAL_CURVE.exists():
    val_curve = pd.read_csv(EXP40_VAL_CURVE)
    missing = expected_cols - set(val_curve.columns)
    if missing:
        raise RuntimeError(
            f"Exp40 validation curve exists but is missing columns: {sorted(missing)}"
        )
    print("[CACHE HIT] reused Exp40 validation beta curves")
else:
    print("[CACHE MISS] rebuilding validation beta curves from existing caches")

    rows = []
    for d in DATASETS:
        for h in HORIZONS:
            prr = load_prr(d, h, "val")["retrieval"]

            for b in BACKBONES:
                dv = load_direct(b, d, h, "val")
                direct = dv["direct"]
                truth = dv["truth"]

                dm = mse(direct, truth)

                for beta in BETA_GRID:
                    pred = combine(direct, prr, beta)
                    mm = mse(pred, truth)

                    rows.append({
                        "backbone": b,
                        "dataset": d,
                        "horizon": h,
                        "beta": float(beta),
                        "direct_val_mse": dm,
                        "calibrated_val_mse": mm,
                        "mse_ratio_vs_direct": mm / dm,
                        "gain_vs_direct_pct": gain_pct(dm, mm),
                    })

    val_curve = pd.DataFrame(rows)

assert len(val_curve) == 120 * len(BETA_GRID)
assert val_curve.notna().all().all()

# Ensure this notebook uses exactly the intended grid.
actual_grid = sorted(val_curve["beta"].unique().tolist())
assert np.allclose(actual_grid, BETA_GRID.tolist()), (actual_grid, BETA_GRID.tolist())

display(val_curve.head(14))
print("[PASS] validation beta curves ready")


In [ ]:

# Cell 7 — select one beta per condition using VALIDATION ONLY

keys = ["backbone", "dataset", "horizon"]

selected = (
    val_curve
    .sort_values(
        keys + ["calibrated_val_mse", "beta"],
        ascending=[True, True, True, True, True],
    )
    .groupby(keys, as_index=False)
    .first()
    .rename(columns={
        "beta": "selected_beta",
        "calibrated_val_mse": "selected_val_mse",
        "gain_vs_direct_pct": "selected_val_gain_pct",
        "mse_ratio_vs_direct": "selected_val_mse_ratio",
    })
)

assert len(selected) == 120
assert selected["selected_beta"].isin(BETA_GRID).all()

print("Selected beta distribution")
beta_dist = (
    selected.groupby("selected_beta", as_index=False)
    .size()
    .rename(columns={"size": "conditions"})
)
display(beta_dist)

print("\nValidation selection summary")
print("mean validation gain (%):", selected["selected_val_gain_pct"].mean())
print("median validation gain (%):", selected["selected_val_gain_pct"].median())
print("beta=0 conditions       :", int((selected["selected_beta"] == 0).sum()))
print("beta>0 conditions       :", int((selected["selected_beta"] > 0).sum()))

display(
    selected[
        ["backbone", "dataset", "horizon",
         "selected_beta", "selected_val_gain_pct"]
    ].head(30)
)

selected.to_csv(OUT / "selected_condition_betas.csv", index=False)
beta_dist.to_csv(OUT / "selected_beta_distribution.csv", index=False)

print("[PASS] 120 condition-specific betas selected using validation only")


In [ ]:

# Cell 8 — TEST evaluation with frozen condition-specific betas

rows = []

for r in selected.itertuples(index=False):
    b = r.backbone
    d = r.dataset
    h = int(r.horizon)
    beta = float(r.selected_beta)

    prr = load_prr(d, h, "test")["retrieval"]
    dt = load_direct(b, d, h, "test")

    direct = dt["direct"]
    truth = dt["truth"]

    dm = mse(direct, truth)
    dmae = mae(direct, truth)

    fixed_pred = combine(direct, prr, FIXED_BETA)
    fixed_mse = mse(fixed_pred, truth)
    fixed_mae = mae(fixed_pred, truth)

    cal_pred = combine(direct, prr, beta)
    cal_mse = mse(cal_pred, truth)
    cal_mae = mae(cal_pred, truth)

    rows.append({
        "backbone": b,
        "dataset": d,
        "horizon": h,

        "selected_beta": beta,
        "selected_val_gain_pct": float(r.selected_val_gain_pct),

        "direct_mse": dm,
        "fixed01_mse": fixed_mse,
        "condition_cal_mse": cal_mse,

        "direct_mae": dmae,
        "fixed01_mae": fixed_mae,
        "condition_cal_mae": cal_mae,

        "fixed01_gain_vs_direct_pct": gain_pct(dm, fixed_mse),
        "condition_cal_gain_vs_direct_pct": gain_pct(dm, cal_mse),
    })

test120 = pd.DataFrame(rows)

assert len(test120) == 120
assert test120.notna().all().all()

test120.to_csv(OUT / "test_120_condition_calibrated.csv", index=False)
display(test120.head(24))

print("[PASS] condition-calibrated test evaluation = 120/120")


In [ ]:

# Cell 9 — sanity check: reproduce fixed beta=.1 result

fixed_counts = wtl(test120["fixed01_gain_vs_direct_pct"])

print("Fixed α=.1 W/T/L:", fixed_counts)
print("Fixed α=.1 mean gain:",
      test120["fixed01_gain_vs_direct_pct"].mean())
print("Fixed α=.1 median gain:",
      test120["fixed01_gain_vs_direct_pct"].median())

if EXP40_TEST120.exists():
    exp40 = pd.read_csv(EXP40_TEST120)

    needed = {
        "backbone", "dataset", "horizon",
        "direct_mse", "fixed_0p1_mse",
    }

    if needed.issubset(exp40.columns):
        chk = test120.merge(
            exp40[[
                "backbone", "dataset", "horizon",
                "direct_mse", "fixed_0p1_mse",
            ]],
            on=["backbone", "dataset", "horizon"],
            suffixes=("_exp41", "_exp40"),
        )
        assert len(chk) == 120

        max_direct_diff = float(
            np.max(np.abs(chk["direct_mse_exp41"] - chk["direct_mse_exp40"]))
        )
        max_fixed_diff = float(
            np.max(np.abs(chk["fixed01_mse"] - chk["fixed_0p1_mse"]))
        )

        print("Max Direct diff vs Exp40:", max_direct_diff)
        print("Max fixed .1 diff vs Exp40:", max_fixed_diff)

        assert max_direct_diff < 5e-6
        assert max_fixed_diff < 5e-6

        print("[PASS] Exp41 reproduces Exp40 fixed-beta values")
    else:
        print("[INFO] Exp40 test file present but expected columns differ; skip cross-check.")
else:
    print("[INFO] Exp40 test_120_conditions.csv not found; internal fixed-beta result shown above.")


In [ ]:

# Cell 10 — overall result

condition_gain = test120["condition_cal_gain_vs_direct_pct"]
fixed_gain = test120["fixed01_gain_vs_direct_pct"]

overall = pd.DataFrame([
    {
        "scheme": "Fixed-0.1",
        **wtl(fixed_gain),
        "mean_gain_vs_direct_pct": float(fixed_gain.mean()),
        "median_gain_vs_direct_pct": float(fixed_gain.median()),
        "min_gain_pct": float(fixed_gain.min()),
        "max_gain_pct": float(fixed_gain.max()),
    },
    {
        "scheme": "Condition-Cal",
        **wtl(condition_gain),
        "mean_gain_vs_direct_pct": float(condition_gain.mean()),
        "median_gain_vs_direct_pct": float(condition_gain.median()),
        "min_gain_pct": float(condition_gain.min()),
        "max_gain_pct": float(condition_gain.max()),
    },
])

display(overall)
overall.to_csv(OUT / "overall_summary.csv", index=False)


In [ ]:

# Cell 11 — dataset / backbone / horizon summaries

def grouped_summary(group_cols):
    rows = []
    for key, g in test120.groupby(group_cols, sort=False):
        if not isinstance(key, tuple):
            key = (key,)

        gains = g["condition_cal_gain_vs_direct_pct"]
        row = dict(zip(group_cols, key))
        row.update(wtl(gains))
        row.update({
            "conditions": len(g),
            "mean_gain_vs_direct_pct": float(gains.mean()),
            "median_gain_vs_direct_pct": float(gains.median()),
            "mean_selected_beta": float(g["selected_beta"].mean()),
            "beta0_conditions": int((g["selected_beta"] == 0).sum()),
        })
        rows.append(row)

    return pd.DataFrame(rows)

dataset_summary = grouped_summary(["dataset"])
backbone_summary = grouped_summary(["backbone"])
horizon_summary = grouped_summary(["horizon"])

print("DATASET")
display(dataset_summary)

print("BACKBONE")
display(backbone_summary)

print("HORIZON")
display(horizon_summary)

dataset_summary.to_csv(OUT / "dataset_summary.csv", index=False)
backbone_summary.to_csv(OUT / "backbone_summary.csv", index=False)
horizon_summary.to_csv(OUT / "horizon_summary.csv", index=False)


In [ ]:

# Cell 12 — calibration generalization audit

# Does a larger validation gain actually predict a larger test gain?
audit = test120[[
    "backbone", "dataset", "horizon",
    "selected_beta",
    "selected_val_gain_pct",
    "condition_cal_gain_vs_direct_pct",
]].copy()

audit["val_positive"] = audit["selected_val_gain_pct"] > 0
audit["test_positive"] = audit["condition_cal_gain_vs_direct_pct"] > TOL
audit["generalized_sign"] = audit["val_positive"] == audit["test_positive"]

corr = float(
    audit["selected_val_gain_pct"].corr(
        audit["condition_cal_gain_vs_direct_pct"]
    )
)

print("Validation/test gain Pearson correlation:", corr)
print("Same improvement sign:",
      int(audit["generalized_sign"].sum()), "/", len(audit))

print("\nBeta-specific test behavior")
beta_test = (
    test120.groupby("selected_beta", as_index=False)
    .agg(
        conditions=("dataset", "size"),
        mean_val_gain_pct=("selected_val_gain_pct", "mean"),
        mean_test_gain_pct=("condition_cal_gain_vs_direct_pct", "mean"),
        median_test_gain_pct=("condition_cal_gain_vs_direct_pct", "median"),
    )
)
display(beta_test)

audit.to_csv(OUT / "calibration_generalization_audit.csv", index=False)
beta_test.to_csv(OUT / "beta_specific_test_behavior.csv", index=False)


In [ ]:

# Cell 13 — paper-facing decision block

fixed_row = overall[overall["scheme"] == "Fixed-0.1"].iloc[0]
cal_row = overall[overall["scheme"] == "Condition-Cal"].iloc[0]

summary = {
    "experiment": "Exp41",
    "training_performed": False,
    "forecaster_inference_performed": False,
    "test_used_for_beta_selection": False,
    "selection_unit": "dataset × horizon × backbone",
    "selection_metric": "validation MSE",
    "beta_grid": BETA_GRID.tolist(),

    "fixed_beta_0p1": {
        "W": int(fixed_row["W"]),
        "T": int(fixed_row["T"]),
        "L": int(fixed_row["L"]),
        "mean_gain_pct": float(fixed_row["mean_gain_vs_direct_pct"]),
        "median_gain_pct": float(fixed_row["median_gain_vs_direct_pct"]),
    },

    "condition_cal": {
        "W": int(cal_row["W"]),
        "T": int(cal_row["T"]),
        "L": int(cal_row["L"]),
        "mean_gain_pct": float(cal_row["mean_gain_vs_direct_pct"]),
        "median_gain_pct": float(cal_row["median_gain_vs_direct_pct"]),
        "beta0_conditions": int((test120["selected_beta"] == 0).sum()),
        "beta_positive_conditions": int((test120["selected_beta"] > 0).sum()),
    },

    "validation_test_gain_correlation": float(corr),
}

(OUT / "exp41_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))

print("\n" + "=" * 100)
print("INTERPRETATION GUIDE")
print("=" * 100)
print(
    "Strong candidate for the main downstream result only if Condition-Cal "
    "shows a clear improvement over Fixed-0.1 in both win count and mean gain, "
    "without obvious dataset/backbone collapse."
)
print(
    "Because every beta is chosen from validation only and then frozen on test, "
    "this is a legitimate condition-specific hyperparameter calibration, not test tuning."
)
print(
    "If the gain is marginal or unstable, stop here and keep Fixed-0.1 as the main result."
)
print("\nSaved under:", OUT)
